# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 제출 정보
- 이름: 김지인
- GitHub ID: jiin-0
- 작성일: 2026.10.14
- 최종 제출 URL: https://github.com/jiin-0/llm-data-analysis-study/blob/main/chapter05/ch05_data_preprocessing.ipynb

## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [8]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == 'notebooks':
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: c:\dev\llm-data-analysis-study\notebooks\ch05
프로젝트 루트: c:\dev\llm-data-analysis-study\notebooks\ch05
원본 데이터 폴더: c:\dev\llm-data-analysis-study\notebooks\ch05\data\raw
전처리 데이터 폴더: c:\dev\llm-data-analysis-study\notebooks\ch05\data\processed
보고서 폴더: c:\dev\llm-data-analysis-study\notebooks\ch05\reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [9]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [10]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,152,6
1,products,102,4
2,orders,302,5
3,order_items,765,5


In [11]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (152, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (102, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (302, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (765, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



In [34]:
print(customers.info())
print(orders.info())

<class 'pandas.DataFrame'>
RangeIndex: 152 entries, 0 to 151
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   customer_id  152 non-null    int64
 1   name         152 non-null    str  
 2   gender       152 non-null    str  
 3   age          148 non-null    str  
 4   city         150 non-null    str  
 5   signup_date  152 non-null    str  
dtypes: int64(1), str(5)
memory usage: 11.7 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 302 entries, 0 to 301
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   order_id        302 non-null    int64
 1   customer_id     302 non-null    int64
 2   order_date      302 non-null    str  
 3   payment_method  302 non-null    str  
 4   order_status    302 non-null    str  
dtypes: int64(2), str(3)
memory usage: 20.0 KB
None


In [35]:
print(order_items.info())
print(products.info())

<class 'pandas.DataFrame'>
RangeIndex: 765 entries, 0 to 764
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_item_id  765 non-null    int64
 1   order_id       765 non-null    int64
 2   product_id     765 non-null    int64
 3   quantity       765 non-null    str  
 4   unit_price     765 non-null    str  
dtypes: int64(3), str(2)
memory usage: 34.9 KB
None
<class 'pandas.DataFrame'>
RangeIndex: 102 entries, 0 to 101
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   product_id    102 non-null    int64
 1   product_name  102 non-null    str  
 2   category      102 non-null    str  
 3   price         102 non-null    str  
dtypes: int64(1), str(3)
memory usage: 6.5 KB
None


## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [12]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [13]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum())



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            4
city           2
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [14]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values('missing_count', ascending=False)

missing_summary(customers_clean)


,missing_count,missing_ratio
age,4,2.63
city,2,1.32
name,0,0.00
customer_id,0,0.00
gender,0,0.00
signup_date,0,0.00


## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [15]:
if 'age' in customers_clean.columns:
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')
    age_median = customers_clean['age'].median()
    customers_clean['age'] = customers_clean['age'].fillna(age_median)
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')

missing_summary(customers_clean)


age 중앙값: 41.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [16]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum())


customers 전체 행 중복 수: 2
products 전체 행 중복 수: 2
orders 전체 행 중복 수: 2
order_items 전체 행 중복 수: 2


In [17]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 2
products product_id 중복 수: 2
orders order_id 중복 수: 2
order_items order_item_id 중복 수: 2


#### 가이드 파일을 기준으로, 어디에 이 md를 넣어야 할지 모호해 해당위치에 삽입하였습니다.

## 1. 전처리 전 상태 기록
- 각 데이터 shape: customers: (152, 6), products: (102,4), orders:(302, 5), order_items:(765, 5)
- 결측치: customers의 age에 4개, customers의 city에 2개의 결측이 각각 존재한다.
- 전체 중복: customers, products, orders, order_items에서 각각 전체 행 중복 수: 2
- 주요 ID 중복: customers customer_id 중복 수: 2, products product_id 중복 수: 2, orders order_id 중복 수: 2, order_items order_item_id 중복 수: 2
- 타입 문제 후보: customers-age/signup_date, orders-order_date, order_items-quantity/unit_price, products-price  (해당 column들이 string으로 인식되고 있다.)

![전처리 전 상태](images/step01_before.png)

### 결과 관찰
customers는 152행과 6열, products는 102행과 4열, orders는 302행과 5열, order_items는 765행과 5열이 관찰되었다.

결측은 customers에서만 관찰되었으며 세부적으로 age에서 4개, city에서 2개로 나타났다.

전체적으로 모든 파일에서 중복행이 2개 관찰되었다.

타입은 customers-age/signup_date, orders-order_date, order_items-quantity/unit_price, products-price이 모두 string으로 인식되고 있어 문제 후보로 선정하였다.


### 나의 해석과 판단
우선 전체 중복행을 삭제하고, 그 후로 id 중복을 재확인한 뒤 이를 해결해야한다. 또한 그 후 각 column들의 type이 기대와 다른 부분들을 변환하고, 각 column의 결측치를 채워야 한다고 생각한다.

그 까닭은, 전체 중복을 처리한 후에 id 중복이 사라질 수 있기 때문이다. 또한, 전체 중복이 가장 큰 범위에서의 데이터 오류이므로 큰 범주에서 출발하는 것이 좋다고 판단했다. 따라서 전체 중복을 처리하고 id 중복을 확인하여 문제가 여전히 남아 있을 경우 추가적인 처리를 해야한다고 생각했다. 다만 이경우, id는 고유한 값이므로 중복행 제거에서 없어지지 않으면 raw data를 신뢰하기 어려워질 수 있다.

이와 같이 중복을 정리한 후, 결측을 처리하기 위해 column의 타입을 원하는 타입으로 각각 필요한 경우 변환해주고, 결측행을 완전히 지울지, 또는 통계값을 이용해 채울지 결정하여 결측치를 해결해야한다고 생각한다.

### 업무·분석적 의미
이와 같이 전처리전 데이터를 확인하여 어떤 전처리가 필요한지 가늠하고, 전처리의 방향을 설계하는 것이 필요하다. 이 과정을 통해 데이터 상태를 보다 자세히 알 수 있고, 데이터 전처리 계획을 각 데이터에 맞게 설정할 수 있다.

### 한계와 추가 확인 사항
전체 중복행을 처리한 후 id가 중복인지 확인이 필요하다.

In [18]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


In [38]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')

customers customer_id 중복 수: 0
products product_id 중복 수: 0
orders order_id 중복 수: 0
order_items order_item_id 중복 수: 0


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


In [36]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),
            result[col].astype(str).str.strip()
        )

    return result

customers_clean = strip_string_columns(customers_clean)
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


C:\Users\1025j\AppData\Local\Temp\ipykernel_33512\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\1025j\AppData\Local\Temp\ipykernel_33512\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/us

In [37]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
부산    23
대구    21
광주    21
서울    18
울산    17
인천    16
수원    10
성남     9
고양     8
대전     5
Name: count, dtype: int64

order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64

## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [21]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [22]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-12,2026-07,Sunday
1,2025-10-02,2025-10,Thursday
2,2026-03-13,2026-03,Friday
3,2025-10-12,2025-10,Sunday
4,2026-04-17,2026-04,Friday


In [23]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


signup_date 변환 실패: 2


,customer_id,signup_date
0,1,2025-10-05
1,2,2023-09-20
2,3,2025-01-22
3,4,2025-03-13
4,5,2024-02-10


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [24]:
def to_number(series):
    return pd.to_numeric(
        series.astype(str).str.replace(',', '', regex=False),
        errors='coerce'
    )

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


price 변환 실패: 2
quantity 변환 실패: 5
unit_price 변환 실패: 2


## 2. 문자열·타입·결측 처리
- 적용한 문자열 정리: 앞뒤 공백 제거, 표현 통일(대소문자, 영어 및 한글)
- 숫자 변환 대상과 실패 건수: customers의 age(실패 count없이 바로 중간값 처리), order_items의 quantity(실패 5개), unit_price(실패 2개), products의 price(실패 2개)
- 날짜 변환 대상과 실패 건수: custonmers의 signup_date(실패 2개), orders의 order_date(실패 3개)
- 결측 처리 기준: age-숫자 변환 후 중간값/ city-Unknown

![타입과 결측 처리](images/step02_cleaning.png)

### 결과 관찰
문자열 앞뒤로 공백을 제거하였으며, 대소문자 및 영어 혹은 한글 표현을 통일하였다. 또한, 앞서 형식이 잘못 인식된 column들을 변환하였고, 그 과정에서 quantity에서 5개, unit_price 2개, price 2개, signup_date 2개, order_date 3개를 각각 변환에 실패했다.

### 나의 해석과 판단
왜 해당 처리 방식을 선택했는지 작성하세요. 삭제/대체가 정보 손실을 만들 가능성도 설명하세요.
문자열의 경우 인식에 있어 사람의 눈에는 앞뒤 공백이 있어도 같게 보일 수 있지만 데이터를 읽는 관점에서는 완전히 동일하지는 않은 값으로 인식될 수 있다. 따라서 앞뒤 공백을 제거했다.

또한, age 결측을 처리할 때, age를 숫자열로 변환한 후 중앙값으로 채워넣었다. 이는 원래의 나이 분포를 붕괴하지 않으면서 데이터를 살리기 위한 것으로, 나이에 대해서만 통계처리한 것으로 볼 수 있다.

city 결측은 city 자체가 string type이기 때문에 통계처리를 하지 않고 unknown처리 하였다. 이를 통해 age에서는 오류가 발생하지 않게 통계를 준수하는 선에서 수를 채우고, city는 type이 string이라는 점을 고려해 특정 값으로 채우지 않고, 결측 내용을 유지하되, 빈칸으로 두지는 않는 처리를 진행했다.

그외 숫자와 날짜 변환이 필요한 column들을 변환 진행하였다. 이를 통해 추후 진행될 파생컬럼 생성에서 발생할 수 있는 오류를 방지할 수 있고, 데이터 분석에서 발생할 수 있는 오류를 사전에 막을 수 있다.

현재 날짜 변환 결과 가장 늦은 날짜가 28년이므로 미래이므로 데이터에 오류가 존재한다.

### 업무·분석적 의미
결측을 처리하여 추후 분석에서 결측으로 인해 발생할 수 있는 계산 실패나 분석 오류를 줄일 수 있다. 열들의 형식 변환을 통해 data를 분석에 용이하도록 가공하였다. 이를 통해 분석의 정확성을 높일 수 있다.

### 한계와 추가 확인 사항
age 결측을 처리할 때, age를 숫자열로 변환한 후 중앙값으로 채워넣었다. 이는 원래의 나이 분포를 붕괴하지 않으면서 데이터를 살리기 위한 것으로, 나이에 대해서만 통계처리한 것으로 볼 수 있다. 따라서 개별 고객의 특성이 반영되지 않은 값이다. 해당 고객으로 나이와 소비패턴을 분석한다면 정확도가 낮아질 수 있으므로 유의해야한다. 해당 고객을 필요한 경우 중앙값 처리했다는 표지를 해두면 좋을 거 같다.

또한, 변환에 실패한 값은 실제 해당 행을 불러와 값을 직접 확인해볼 필요가 있다. 그러나 처리과정에서 결측으로 수정해 확인이 불가하다.

## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [39]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count    9.800000e+01
mean     2.005102e+05
std      1.001829e+06
min     -2.000000e+04
25%      5.200000e+04
50%      9.250000e+04
75%      1.527500e+05
max      9.999999e+06
Name: price, dtype: float64

,quantity,unit_price
count,758.000000,761.000000
mean,3.168865,103494.086728
std,4.159055,56969.823769
min,-1.000000,0.000000
25%,2.000000,55000.000000
50%,3.000000,96000.000000
75%,4.000000,154000.000000
max,100.000000,197000.000000


In [47]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 2
quantity <= 0: 3
unit_price <= 0: 1


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [48]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (96, 4)
이상값 처리 후 order_items: (752, 5)


## 3. 중복·범주 표준화·이상값 후보
- 제거/보류한 중복: id 중복과 전제 행 중복 중 id 중복은 보류하고, 우선적으로 전체 행 중복을 제거하였다. 이후 id 중복이 사라져 추가적 중복 제거를 진행하지 않았다.
- 표준화한 범주값: completed, cancelled, refunded (order_status)
- 허용값 밖 값: price <= 0,quantity <= 0, unit_price <= 0
- 이상값 후보: price-2개, quantity-3개, unit_price-1개

![중복 범주 이상값](images/step03_rules.png)

### 결과 관찰
중복 행을 제거하자 중복 id 또한 제거되었다. 범주값들을 표준화하여 completed, cancelled, refunded로 형식을 맞췄으며, 실습용으로 이상값의 범위를 지정(가격, 양에 대해 0이하)하고, 해당 범위 밖 값이 몇개 있는지 확인(price-2개, quantity-3개, unit_price-1개)하고, 제거하였다.

### 나의 해석과 판단
중복제거를 통해 raw data의 오류를 줄였다.

이상값의 범위와 처리는 분석 목적 및 상황에 따라 달라진다. 따라서 이번 실습에서 설정한 '0이하'라는 기준은 현재 실습에서만 유효할 뿐, 다른 분석에서 바로 적용할 수 없으며, 목적에 맞추어 이상값의 기준을 설정해야한다. 이와 같은 이상값 설정이후, 이상값을 바로 삭제해서는 안된다. 이상값이 통계상으로 혼자 튈 수 있어도, 값이 의미를 가지고 있을 수 있으며, 전처리의 오류로 소수점이 잘못찍혔을 수 있다. 따라서 이상값을 직접확인하고 판단해야한다. 

### 업무·분석적 의미
이상값은 통계의 관점에서 분포를 망가뜨리는 역할을 한다. 따라서 이상값을 확인하고 처리하는 것이 중요하다. 그러나 이상값이 실제 필요하다면 이상값의 영향을 적게 받는 통계값을 선택하여 처리하면 된다. 또한 범주를 표준화하여 추후 이어질 분석에서 발생할 수 있는 오류를 줄일 수 있다.

### 한계와 추가 확인 사항
데이터의 형성 배경을 확인하여 현재 범위 내의 이상값의 의미를 알아볼 필요가 있다.

## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [49]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [50]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 252140000.0


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [51]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


## 4. PK/FK와 파생 컬럼 재검증
- 없는 `customer_id`: 2건
- 없는 `order_id`: 2건
- 없는 `product_id`: 26건
- 생성한 파생 컬럼: line_total
- `line_total` 검증: line_total.sum()

![관계와 파생 컬럼 검증](images/step04_validation.png)

### 결과 관찰
이상값을 제거하고, 파생컬럼을 만든 뒤 id를 비교한 결과 없는 customer_id가 2건, 없는 order_id가 2건, 없는 product_id가 26건 관찰되었다.

### 나의 해석과 판단
이처럼 전처리 전에 연결이 되던 key가 전처리 후로 key 연결이 끊길 수 있다. 따라서 전처리 전후로 키관계 확인이 필수적이다.

또한, 현재 customer_id 매치가 되지 않는 것은 원래 data 상에서부터 매치가 되지 않은 값일 것이다. customer에서는 중복만 제거하였으므로 원래있던 값을 전체를 지우지 않았기 때문이다.

## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [52]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,products,102,4,96,4
2,orders,302,5,300,7
3,order_items,765,5,752,6


## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [53]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch05/data/processed/customers_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch05/data/processed/orders_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch05/data/processed/order_items_clean.csv'),
 WindowsPath('c:/dev/llm-data-analysis-study/notebooks/ch05/data/processed/products_clean.csv')]

## 5. 전처리 전/후 비교
customers
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 |152 |150 | 중복 제거(2)|
| 결측 | age 4 city 2| 0| 결측 처리 완료|
| 중복 |customer_id 2 |0 | 중복 전체행 제거 후 해결|
| 변환 실패 | 0 | 2 | 형식 변환 이후 signup_date에서 날짜형으로 2건 변환 실패 |

products
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 102| 96| 중복 제거(2), price 0이하 삭제(2), price 숫자형 변환 실패행 삭제(2)|
| 결측 | 0| 0| X|
| 중복 | products id 2| 0| 중복 전체행 제거 후 해결|
| 변환 실패 | 0| 0 | 변환 실패 행이 있었으나 이상값 제거하면서 함께 제거|

orders
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 302|300 |중복 제거(2) |
| 결측 |0 |0 |X |
| 중복 |order_id 2 | 0| 전체 중복행 제거 후 해결|
| 변환 실패 |0 |3 | order_date 날짜형 변환 과정에서 변환 실패|

order_items
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 765 |752 |중복 제거(2), quantity 0이하 삭제(3), quantity 숫자형 변환 실패 삭제(5), unit_price 0이하 삭제(1), unit_price 숫자형 변환 실패 삭제(2) |
| 결측 |0 | 0| X|
| 중복 |order_item_id 2 | X|전체 중복행 제거 후 해결 |
| 변환 실패 |0 |0 | 변환 실패 데이터가 있었으나 이상 값처리 과정에서 삭제됨.|

![전처리 전후 비교](images/step05_before_after.png)

### 결과 관찰
모든 파일에서 행 수가 감소하였다. 또한, orders와 order_item은 열이 증가하였는데 orders는 order_date와 관련하여 월과 요일 열이 추가되었고, order_item은 line_total 열이 추가되었다. 자세한 변화는 상단의 표들에 기재해두었다.

### 나의 해석과 판단
각 파일에서 행 수의 감소는 분석 목적에 부합하는 부분과 부합하지 않을 수도 있는 부분이 존재한다.
전체 행 중복은 deduplicate 명령어를 이용해 제거하였으므로, 의도에 부합한다. 그러나 이상값 제거 과정에서 형태 변환에 실패한 값들 또한 0보다 큰 값이 아니므로 제거되었다. 이는 변환 오류로 인한 결측으로 인해 삭제된 것으로, 원래의 의도인 이상값과는 다르게 삭제된 것이다.

### 업무·분석적 의미
이러한 전후 변화를 관찰하고 증감 요인을 분석하며 데이터 전처리 결과와 과정을 명확히 할 수 있다. 또한 이어지는 데이터 분석에서의 오류 발생 시 해결시작점을 잡는데 유용한 정보가 된다.


### 한계와 추가 확인 사항
현재 date관련 변환 실패 결측치를 처리하지 않았다. 또한, 의도한 바와 다르게 숫자형 데이터 변환 실패 값들을 가지고 있는 행들도 삭제되었다. 따라서 변환 과정에서 값들을 실패값들을 직접확인하고 이러한 문제를 해결할 필요가 있다.

*분석 목적이 날짜가 포함되지 않는 경우 불필요하다. 그러나 숫자형 데이터는 필요하므로 google sheet의 기능을 활용하거나 혹은 직접 오류가 발생한 행을 찾고, raw 파일을 열어 확인해보면 된다.


In [69]:
products_clean[products_clean['price'].isna()] #이와 같은 코드를 활용해 products_id를 기준으로 raw 파일에서 찾아보면 된다.
# 그러나 현재 이미 관련행이 삭제되어 확인이 불가능하다.

,product_id,product_name,category,price


## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [54]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: c:\dev\llm-data-analysis-study\notebooks\ch05\reports\ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [55]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    return result.drop_duplicates()


In [56]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(96, 4)
(300, 7)
(752, 6)


C:\Users\1025j\AppData\Local\Temp\ipykernel_33512\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns
C:\Users\1025j\AppData\Local\Temp\ipykernel_33512\1976643875.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/us

## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [115]:
from pathlib import Path
import sys

# 현재 폴더부터 상위 폴더를 올라가며 src/data_loader.py 찾기
project_root = Path.cwd().resolve()

while project_root != project_root.parent:
    if (project_root / "src" / "data_loader.py").exists():
        break
    project_root = project_root.parent

if not (project_root / "src" / "data_loader.py").exists():
    raise FileNotFoundError(
        "src/data_loader.py를 찾지 못했습니다. 현재 노트북이 올바른 프로젝트 폴더에서 실행 중인지 확인하세요."
    )

# 프로젝트 루트를 파이썬 모듈 검색 경로에 추가
sys.path.insert(0, str(project_root))

In [112]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,order_items,765,5,752,6
2,orders,302,5,300,7
3,products,102,4,96,4


In [113]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [114]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


## 6. 재현 가능한 전처리 확인
- 생성된 clean CSV: customers_clean.csv, order_items_clean.csv, orders_clean.csv, products_clean.csv
- 생성된 요약 보고서: ch05_preprocessing_summary.md
- `python scripts/preprocess_data.py` 재실행 결과: clean.csv 파일들과 summary.md 파일이 생성되었다.
- 재실행 후 달라진 점: 없다. 똑같이 파일이 생성되었다.

![재실행 결과](images/step06_reproducible.png)

### 나의 해석과 판단
노트북은 셀 단위로 코드를 실행하기 때문에 순서가 뒤죽박죽이어도 일시적으로 성공할 수 있다. 반면 스크립트는 항상 위에서 아래로 실행되므로 코드가 완벽한 순서로 짜여 있어야만 작동한다. 또한 노트북은 실행하는 사람의 컴퓨터 환경이나 폴더 위치에 따라 경로 에러가 자주 나지만, 스크립트는 프로젝트 루트 경로와 환경 변수(.env)를 고정해 두기 때문에 언제, 어디서, 누가 실행해도 항상 똑같은 결과가 기대된다.

## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


## 24. 실습 과제

아래 과제를 직접 해결해 보세요.
1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. orders_clean에서 월별 주문 건수를 계산하세요.
4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.

In [110]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.
def missing_summary(df):
    summary = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_ratio": (df.isna().mean() * 100).round(2),
    })
    return summary.sort_values("missing_count", ascending=False)


raw_data = {
    "customers": customers,
    "orders": orders,
    "products": products,
    "order_items": order_items,
}

summary_list = []

for raw_data, df in raw_data.items():
    summary = missing_summary(df)
    summary = summary.reset_index()
    summary.insert(0, "dataset", raw_data)

    summary_list.append(summary)

missing_summary_all = pd.concat(summary_list, ignore_index=True)

missing_summary_all

,dataset,index,missing_count,missing_ratio
0,customers,age,4,2.63
1,customers,city,2,1.32
2,customers,name,0,0.00
3,customers,customer_id,0,0.00
4,customers,gender,0,0.00
5,customers,signup_date,0,0.00
6,orders,order_id,0,0.00
7,orders,customer_id,0,0.00
8,orders,order_date,0,0.00
9,orders,payment_method,0,0.00


In [84]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.

print(customers_clean[customers_clean['age']<18])
print(customers_clean[customers_clean["age"]>100])

Empty DataFrame
Columns: [customer_id, name, gender, age, city, signup_date]
Index: []
    customer_id name gender    age city signup_date
32           33  권승윤      F  150.0   고양  2025-09-14


In [103]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.
monthly_order_count = (
    orders_clean.groupby("order_month")["order_id"]
    .nunique()
    .rename("order_count")
    .reset_index()
)

print(monthly_order_count)


   order_month  order_count
0      2025-07           19
1      2025-08           26
2      2025-09           12
3      2025-10           22
4      2025-11           23
5      2025-12           23
6      2026-01           26
7      2026-02           19
8      2026-03           22
9      2026-04           26
10     2026-05           20
11     2026-06           24
12     2026-07           12
13     2026-08           21
14     2027-01            1
15     2028-12            1


In [81]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.
print(order_items_clean.sort_values("line_total", ascending=False).head(10))

     order_item_id  order_id  product_id  quantity  unit_price  line_total
750            751       295          31     100.0    155000.0  15500000.0
737            738       290          96      50.0     96000.0   4800000.0
319            320       129          23       5.0    197000.0    985000.0
9               10         5          65       5.0    193000.0    965000.0
24              25        11          65       5.0    193000.0    965000.0
293            294       119          36       5.0    193000.0    965000.0
116            117        51          65       5.0    193000.0    965000.0
20              21         9          36       5.0    193000.0    965000.0
290            291       117          47       5.0    188000.0    940000.0
491            492       195          99       5.0    187000.0    935000.0


5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.-이번 실습에서.
customers
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 |152 |150 | 중복 제거(2)|
| 결측 | age 4 city 2| 0| 결측 처리 완료|
| 중복 |customer_id 2 |0 | 중복 전체행 제거 후 해결|
| 변환 실패 | 0 | 2 | 형식 변환 이후 signup_date에서 날짜형으로 2건 변환 실패 |

products
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 102| 96| 중복 제거(2), price 0이하 삭제(2), price 숫자형 변환 실패행 삭제(2)|
| 결측 | 0| 0| X|
| 중복 | products id 2| 0| 중복 전체행 제거 후 해결|
| 변환 실패 | 0| 0 | 변환 실패 행이 있었으나 이상값 제거하면서 함께 제거|

orders
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 302|300 |중복 제거(2) |
| 결측 |0 |0 |X |
| 중복 |order_id 2 | 0| 전체 중복행 제거 후 해결|
| 변환 실패 |0 |3 | order_date 날짜형 변환 과정에서 변환 실패|

order_items
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | 765 |752 |중복 제거(2), quantity 0이하 삭제(3), quantity 숫자형 변환 실패 삭제(5), unit_price 0이하 삭제(1), unit_price 숫자형 변환 실패 삭제(2) |
| 결측 |0 | 0| X|
| 중복 |order_item_id 2 | X|전체 중복행 제거 후 해결 |
| 변환 실패 |0 |0 | 변환 실패 데이터가 있었으나 이상 값처리 과정에서 삭제됨.|

6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.

온라인 쇼핑몰 데이터의 결측치 처리 기준을 검토할 것.

데이터셋은 customers, orders, products, order_items로 구성되어 있음.
각 데이터셋에는 고객 정보, 주문 정보, 상품 정보, 주문 상품별 수량과 단가가 들어 있음.

아래에 데이터셋별 컬럼명, 결측치 개수와 비율, 각 컬럼의 의미를 참고할 것.

| data set | 컬럼명 | 결측치 개수 | 비율 | 컬럼의 의미 |
| --- | ---: | ---: | ---: | --- |
| customers | age | 4 |2.63 | 나이 |
| customers | city | 2 | 1.32| 거주지역 |

다음 기준으로 검토할 것.
1. 결측 행을 삭제해도 되는 컬럼과 삭제하면 분석에 문제가 생길 수 있는 컬럼을 구분.
2. 평균·중앙값·최빈값 대체가 적절한 컬럼이 있는지 판단.
3. 결측값을 별도 범주(예: '미상')로 처리하는 편이 나은 컬럼이 있는지 검토.
4. 주문·고객·상품 식별자처럼 핵심 키에 결측치가 있을 때의 처리 방안을 제시.
5. 처리 기준마다 근거와 분석 결과에 미칠 수 있는 영향을 설명.
6. 제공된 정보만으로 판단할 수 없는 사항은 추측하지 말고, 추가로 필요한 정보를 밝힐 것.

무조건 권장하지 말고, 가능한 선택지와 장단점을 비교할 것.

## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.


## 7. Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1. 전처리 전에 파일 상태 확인 꼭 할것.(info())
2. 결측 함부로 삭제하지 말 것.
3. 데이터 범위 확인할 것.

### 가장 위험하다고 생각한 자동 처리 1가지와 이유
이상값 처리이다. 긍정의 방향으로 이상값을 처리하니 원치 않은 결측값들도 같이 삭제되었기 때문이다.

### 다음 EDA에서 특히 주의할 데이터 특성
현재 형식 변환에 실패한 값들이 있기 때문에 EDA전 분석 목표가 정해지면 raw파일을 열어 현재의 전처리를 보완해야한다.(미완)

### 현재 전처리의 한계
변환에 실패한 값들을 수정하지 않고 그냥 전처리하였다. 따라서 해당 값들의 손실이 존재한다.
또한, 나이를 중앙값으로 처리하여 고객의 특성이 반영되지 않았다.

## 최종 제출 체크
- [O] raw 원본을 덮어쓰지 않았습니다.
- [O] 처리 전/후 비교를 남겼습니다.
- [O] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [O] clean CSV와 재실행 결과를 확인했습니다.
- [O] 개인정보/Secret이 없습니다.
- [O] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [O] 최종 Notebook 파일 URL을 제출합니다.